WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)

### Geom

In [ ]:
rod_length = 2.0
num_elems = 60
num_nodes = num_elems + 1
fiber_offset = 0.1  # c [m]
fiber_radius = 0.02

In [ ]:
# Four fibers at (+/-c, 0) and (0, +/-c).
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = offsets_local.shape[0]
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

### BC

In [ ]:
num_load_steps = 10

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_load_steps)]),
    node_targets=[np.array([0]) for _ in range(num_load_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

boundary_conditions = _bc.process_boundary_conditions(rod, clamp)

### Magnetic (force)

In [ ]:
remnant_flux = 0.1  # B_r [T], along the rod axis
applied_flux = 9.4e-3  # B_a [T], transverse (global frame)

In [ ]:
remnant_mag_flux = jnp.asarray(
  np.tile(np.array([remnant_flux, 0.0, 0.0]), (num_elems, num_fibers, 1))
)

applied_mag_flux = jnp.asarray(
  np.outer(
    np.linspace(applied_flux / num_load_steps, applied_flux, num_load_steps),
    np.array([0.0, 1.0, 0.0]),
  )
)
print(f"remnant_mag_flux {remnant_mag_flux.shape}, applied {applied_mag_flux.shape}")

### Solver

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-6,
  "abs_threshold": 1e-6,
  "lam_min": 0.01,
  "line_search_max_iter": 15,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}

linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-12,
}

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings=nonlinear_settings,
  linear_settings=linear_settings,
)

problem = _fe.CosseratProblem(
  rod=rod,
  bc=boundary_conditions,
  nr_solver=nr_solver,
)

In [ ]:
shear_modulus = 1.0e6  # G [Pa]
shear_moduli = 1.0e6 * jnp.ones((num_elems, num_fibers))

### Solve

In [ ]:
u_star, u_history = problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=shear_moduli,
  remnant_mag_flux=remnant_mag_flux,
  applied_mag_flux=applied_mag_flux,
)

u_nodal = np.asarray(u_star).reshape(num_nodes, rod.num_dofs_per_node)

### Analytical

In [ ]:
youngs_modulus = 3.0 * shear_modulus
area_total = num_fibers * area_per_fiber
moi_11 = np.sum(area_per_fiber * offsets_local[:, 0] ** 2)  # pairs with kappa_2
bending_stiffness = youngs_modulus * moi_11

tip_force = area_total * remnant_flux * applied_flux / _utils.PERMEABILITY_VACUUM
shear_stiffness = shear_modulus * area_total

bending_part = tip_force * rod_length**3 / (3.0 * bending_stiffness)
shear_part = tip_force * rod_length / shear_stiffness


def deflection_exact(node_x):
  return (
    tip_force * node_x**2 * (3 * rod_length - node_x) / (6 * bending_stiffness)
    + tip_force * node_x / shear_stiffness
  )

In [ ]:
w_anal = deflection_exact(rod_length)

print(f"tip w = {u_nodal[-1, 1]:.6e} m,  analytical " f"{w_anal:.6e} m")

In [ ]:
directors = jax.vmap(_utils.rotate_directors)(
  jnp.asarray(u_nodal[:, 3:]), rod.directors_0
)
kinematics = _mesher.compute_discrete_cosserat_kinematics(
  node_coords=rod.node_coords + jnp.asarray(u_nodal[:, :3]),
  directors=directors,
  arc_coords=rod.arc_coords,
)
curvature_num = np.asarray(kinematics.curvature)[:, 1]  # kappa_2
elem_x = np.asarray(rod.elem_centers)[:, 0]
curvature_exact = tip_force * (rod_length - elem_x) / bending_stiffness

print(f"kappa at clamp : {curvature_num[0]:.6e}  vs  {curvature_exact[0]:.6e}")
print(f"kappa at tip   : {curvature_num[-1]:.6e}  vs  {curvature_exact[-1]:.6e}")

### Plot

In [ ]:
node_x = np.asarray(rod.node_coords)[:, 0]
deflection_exact = (
  tip_force / bending_stiffness * (rod_length * node_x**2 / 2.0 - node_x**3 / 6.0)
)

fig, (ax_w, ax_k) = plt.subplots(1, 2, figsize=(11, 4))
ax_w.plot(node_x, deflection_exact, "k-", lw=2, label="analytical")
ax_w.plot(node_x, u_nodal[:, 1], "o", ms=4, mfc="none", label="CROPS")
ax_w.set_xlabel("$x$ [m]")
ax_w.set_ylabel("$w(x)$ [m]")
ax_w.legend()

ax_k.plot(elem_x, curvature_exact, "k-", lw=2, label=r"$m(L-s)/EI$")
ax_k.plot(elem_x, curvature_num, "o", ms=4, mfc="none", label="CROPS")
ax_k.set_xlabel("$x$ [m]")
ax_k.set_ylabel(r"$\kappa_2$ [1/m]")
ax_k.legend()
fig.tight_layout()
plt.show()

In [ ]:
angles = np.linspace(0.0, np.pi, 19)
tip_heights = []
for angle in angles:
  field = np.outer(
    np.linspace(applied_flux / num_load_steps, applied_flux, num_load_steps),
    np.array([np.cos(angle), np.sin(angle), 0.0]),
  )
  u_sweep, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
    remnant_mag_flux=remnant_mag_flux,
    applied_mag_flux=jnp.asarray(field),
  )
  tip_heights.append(np.asarray(u_sweep).reshape(num_nodes, -1)[-1, 1])

plt.plot(np.degrees(angles), tip_heights, "o-", label="CROPS")
plt.plot(
  np.degrees(angles),
  (tip_force * rod_length**3 / (3.0 * bending_stiffness)) * np.sin(angles),
  "k--",
  label=r"linear: $\propto \sin\theta$",
)
plt.xlabel("applied field angle [deg]")
plt.ylabel("tip height [m]")
plt.legend()
plt.show()

### Grad checks

In [ ]:
def tip_height(remnant, applied):
  """Tip y-displacement. Wang's objective, up to sign."""
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    remnant_mag_flux=remnant,
    applied_mag_flux=applied,
  )
  return u_star.reshape(num_nodes, rod.num_dofs_per_node)[-1, 1]

In [ ]:
grad_remnant = jax.grad(tip_height, argnums=0)(remnant_mag_flux, applied_mag_flux)

step = 1e-7
probe = [(0, 0, 0), (0, 0, 1), (num_elems // 2, 2, 0), (num_elems - 1, 3, 1)]
print(f"{'entry':>16} {'jax.grad':>16} {'central FD':>16} {'rel err':>11}")
for idx in probe:
  plus = remnant_mag_flux.at[idx].add(step)
  minus = remnant_mag_flux.at[idx].add(-step)
  fd = (tip_height(plus, applied_mag_flux) - tip_height(minus, applied_mag_flux)) / (
    2.0 * step
  )
  ad = grad_remnant[idx]
  print(
    f"{str(idx):>16} {ad:16.8e} {fd:16.8e} "
    f"{abs(ad - fd) / max(abs(fd), 1e-30):11.2e}"
  )

In [ ]:
grad_applied = jax.grad(tip_height, argnums=1)(remnant_mag_flux, applied_mag_flux)

print(f"{'step':>6} {'d/dBa_x':>16} {'d/dBa_y':>16}")
for step_idx in range(num_load_steps):
  print(
    f"{step_idx:6d} {grad_applied[step_idx, 0]:16.8e} "
    f"{grad_applied[step_idx, 1]:16.8e}"
  )

In [ ]:
tangent = jax.random.normal(jax.random.PRNGKey(0), remnant_mag_flux.shape)
_, directional = jax.jvp(
  lambda r: tip_height(r, applied_mag_flux), (remnant_mag_flux,), (tangent,)
)
from_reverse = jnp.einsum("efd, efd ->", grad_remnant, tangent)

step = 1e-7
fd_directional = (
  tip_height(remnant_mag_flux + step * tangent, applied_mag_flux)
  - tip_height(remnant_mag_flux - step * tangent, applied_mag_flux)
) / (2.0 * step)

print(f"forward  (jvp) : {directional:.10e}")
print(f"reverse  (grad): {from_reverse:.10e}")
print(f"central  (fd)  : {fd_directional:.10e}")